# Phase 1 Pilot Experiments — Confidence-Guided Token Rejection

**Before running:** Runtime > Change runtime type > **A100 GPU**.

This notebook runs the Phase 1 pilot sweep per the proposal §4:
- 1 vanilla baseline at 32 steps (matched step-count comparison)
- 18 rejection configurations: τ ∈ {0.3, 0.5, 0.7} × max_reject ∈ {0.1, 0.2} × metric ∈ {max_prob, entropy, margin}
- Each generates 10,000 samples, evaluated with FID-10K

**Total runtime: ~5 hours** on A100 (sampling ~1h + FID eval ~2h + overhead).

Both the sampling and FID-eval cells are **resumable** — if Colab disconnects, just re-run the cell and it picks up where it left off.

## Strategy
- Generate all NPZs to **local disk** (`/content/pilot-YYYYMMDD/`) for speed
- Evaluate FID on each NPZ, writing results to CSV
- At the end, copy only small artifacts (CSV, logs, heatmaps) to Drive — NPZs stay local (they'd fill Drive and we can regenerate from the CSV anyway)
- Identify top 3-5 configs for Phase 2

## 1. Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Runtime > Change runtime type > A100 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ARPG = "/content/drive/MyDrive/ARPG-assets"
!mkdir -p {DRIVE_ARPG}/results

In [ ]:
import os
os.chdir('/content')
!rm -rf /content/ARPG
!git clone https://github.com/rshahbazov23/comp447-arpg-private.git /content/ARPG
%cd /content/ARPG

# Symlink heavy assets from Drive (weights, eval reference, guided-diffusion)
!ln -sfn {DRIVE_ARPG}/weights weights
!ln -sfn {DRIVE_ARPG}/eval eval
!ln -sfn {DRIVE_ARPG}/external external

!pip install -q transformers einops Pillow tqdm numpy scipy tensorflow

In [ ]:
# Verify everything is in place.
required = [
    "weights/arpg_300m.pt",
    "weights/vq_ds16_c2i.pt",
    "eval/VIRTUAL_imagenet256_labeled.npz",
    "external/guided-diffusion/evaluations/evaluator.py",
    "scripts/run_pilot_sweep.sh",
    "scripts/eval_pilot_sweep.py",
]
for p in required:
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"  [{status}] {p}")

## 2. Define the pilot output directory

We use a dated folder on local disk (fast I/O). NPZs stay local; only small artifacts get copied to Drive at the end.

In [ ]:
from datetime import datetime

# Use a fixed date so re-running after disconnection uses the same folder.
# If you start a new sweep, change this to a new date.
PILOT_DATE = datetime.now().strftime('%Y%m%d')
PILOT_DIR = f"/content/pilot-{PILOT_DATE}"
DRIVE_RESULTS = f"{DRIVE_ARPG}/results/pilot-{PILOT_DATE}"

!mkdir -p {PILOT_DIR} {DRIVE_RESULTS}

print(f"Local pilot dir:  {PILOT_DIR}")
print(f"Drive results:    {DRIVE_RESULTS}")

## 3. Run the sweep (sampling only)

**~1 hour** of wall-clock time on A100. Generates 19 NPZs (1 vanilla + 18 rejection).

Safe to interrupt — re-running this cell will skip configs that already have an NPZ.

In [ ]:
env = {
    'PILOT_DIR':              PILOT_DIR,
    'NUM_SAMPLES':            '10000',
    'STEPS':                  '32',
    'PER_PROC_BATCH_SIZE':    '32',
    'CFG_SCALE':              '5.0',
    'GLOBAL_SEED':            '0',
}
env_prefix = ' '.join(f'{k}={v}' for k, v in env.items())

!{env_prefix} bash scripts/run_pilot_sweep.sh

In [ ]:
# Verify we have 19 NPZs (1 vanilla + 18 rejection)
import glob
vanilla_npzs = glob.glob(f"{PILOT_DIR}/vanilla/*.npz")
rejection_npzs = glob.glob(f"{PILOT_DIR}/rejection/*.npz")

print(f"Vanilla NPZs:   {len(vanilla_npzs)} / 1")
print(f"Rejection NPZs: {len(rejection_npzs)} / 18")

# Disk usage
!du -sh {PILOT_DIR}

## 4. Run FID evaluation on all NPZs

**~2 hours** — each evaluation is ~4–6 minutes.

Writes `results.csv` incrementally. Safe to interrupt — re-running this cell skips rows already in the CSV.

In [ ]:
CSV_PATH = f"{PILOT_DIR}/results.csv"

!python scripts/eval_pilot_sweep.py \
    --pilot-dir {PILOT_DIR} \
    --reference-npz eval/VIRTUAL_imagenet256_labeled.npz \
    --guided-diffusion external/guided-diffusion/evaluations/evaluator.py \
    --out-csv {CSV_PATH}

## 5. Analyse results

Load the CSV, sort by FID, and highlight the top configs.

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

# Drop failed rows for ranking
df_ok = df[df['fid'].notna()].copy()
df_ok = df_ok.sort_values('fid', ascending=True).reset_index(drop=True)

# Isolate the vanilla baseline for reference
vanilla_row = df_ok[df_ok['mode'] == 'vanilla']
vanilla_fid = vanilla_row['fid'].iloc[0] if len(vanilla_row) else None
print(f"Vanilla FID at 32 steps: {vanilla_fid}")
print(f"Total successful evaluations: {len(df_ok)} / {len(df)}")

df_ok.head(10)

In [ ]:
# Show ALL configs, ranked by FID, with improvement vs vanilla
if vanilla_fid is not None:
    df_ok['delta_vs_vanilla'] = df_ok['fid'] - vanilla_fid
    df_ok['pct_vs_vanilla']   = (df_ok['fid'] - vanilla_fid) / vanilla_fid * 100

cols = ['mode', 'metric', 'tau', 'cap', 'fid', 'is_score', 'precision', 'recall']
if 'delta_vs_vanilla' in df_ok.columns:
    cols += ['delta_vs_vanilla', 'pct_vs_vanilla']

df_display = df_ok[cols].copy()
for c in ['fid', 'delta_vs_vanilla']:
    if c in df_display.columns:
        df_display[c] = df_display[c].round(3)
if 'pct_vs_vanilla' in df_display.columns:
    df_display['pct_vs_vanilla'] = df_display['pct_vs_vanilla'].round(2)

df_display

In [ ]:
# Visual: FID by config, grouped by metric
import matplotlib.pyplot as plt

rej = df_ok[df_ok['mode'] == 'rejection'].copy()
if len(rej):
    rej['config'] = rej.apply(
        lambda r: f"τ={r['tau']} cap={r['cap']}", axis=1
    )
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, metric in enumerate(['max_prob', 'entropy', 'margin']):
        sub = rej[rej['metric'] == metric].sort_values(['tau', 'cap'])
        ax.plot(sub['config'], sub['fid'], marker='o', label=metric, linewidth=2)
    if vanilla_fid is not None:
        ax.axhline(vanilla_fid, color='gray', linestyle='--',
                   label=f'vanilla (32 steps) = {vanilla_fid:.2f}')
    ax.set_ylabel('FID-10K (lower is better)')
    ax.set_xlabel('Configuration')
    ax.set_title('Phase 1 pilot results — FID by (τ, cap, metric)')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Pick top 5 rejection configs for Phase 2
top5 = df_ok[df_ok['mode'] == 'rejection'].head(5).copy()
print("=== Top 5 rejection configurations ===\n")
for i, row in top5.iterrows():
    delta = row.get('delta_vs_vanilla', None)
    delta_str = f" ({delta:+.3f} vs vanilla)" if delta is not None else ""
    print(f"  {i + 1}. metric={row['metric']:10s}  "
          f"tau={row['tau']}  cap={row['cap']}  "
          f"FID={row['fid']:.3f}{delta_str}")

top5[['mode', 'metric', 'tau', 'cap', 'fid', 'is_score',
      'precision', 'recall']].round(3)

## 6. Copy results to Drive

Only copy the CSV, JSON logs, and heatmaps — NOT the NPZs (they're 37 GB total and can be regenerated).

In [ ]:
# Copy results.csv
!cp {CSV_PATH} {DRIVE_RESULTS}/results.csv

# Copy JSON logs + heatmaps from each rejection config
!mkdir -p {DRIVE_RESULTS}/logs
!cp {PILOT_DIR}/logs/*.json {DRIVE_RESULTS}/logs/ 2>/dev/null || echo "(no JSON logs)"
!cp {PILOT_DIR}/logs/*_heatmap.png {DRIVE_RESULTS}/logs/ 2>/dev/null || echo "(no heatmaps)"

print(f"\nResults backed up to: {DRIVE_RESULTS}")
!ls -lh {DRIVE_RESULTS}

## 7. (Optional) Keep top-5 NPZs for Phase 2

Phase 2 (full FID-50K at step counts {16, 32, 64}) will regenerate NPZs anyway, so this step is optional.

In [ ]:
# Uncomment to save the top-5 rejection NPZs to Drive for inspection
# !mkdir -p {DRIVE_RESULTS}/top5-npzs
# for _, row in top5.iterrows():
#     src = f"{PILOT_DIR}/rejection/{row['npz']}"
#     if os.path.exists(src):
#         !cp {src} {DRIVE_RESULTS}/top5-npzs/
# !ls -lh {DRIVE_RESULTS}/top5-npzs

## Summary

If the pilot completed successfully, you now have:
- `results.csv` with FID/IS/Precision/Recall for all 19 configs
- Ranked top-5 rejection configs for Phase 2
- Per-config logs and heatmaps

**Next:** Phase 2 — full FID-50K evaluation on the top 3–5 configs at step counts {16, 32, 64} + refinement-pass ablation at k ∈ {0.1, 0.2}. Target: May 3 for the progress report.